In [1]:
import torch
import math
from torch import nn
import torch.nn.functional as F
import sentencepiece as spm
from torch.optim.lr_scheduler import LambdaLR

## UTILS ##

In [2]:
'''[utils.py]'''
def get_device():
    return torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_sequence_length):
        super().__init__()
        self.d_model = d_model
        self.max_sequence_length = max_sequence_length

        # Precompute positional encoding
        pe = self._compute_pe()
        self.register_buffer("pe", pe)  # not a parameter, but moves with model

    def _compute_pe(self):
        even_i = torch.arange(0, self.d_model, 2).float()
        denominator = torch.pow(10000, even_i / self.d_model)

        position = torch.arange(self.max_sequence_length).reshape(self.max_sequence_length, 1)

        even_PE = torch.sin(position / denominator)
        odd_PE = torch.cos(position / denominator)

        stacked = torch.stack([even_PE, odd_PE], dim=2)
        PE = torch.flatten(stacked, start_dim=1, end_dim=2)

        return PE.unsqueeze(0)  # (1, seq_len, d_model)

    def forward(self):
        return self.pe

class SentenceEmbedding(nn.Module):
    def __init__(self, max_sequence_length, d_model, tokenizer_path, drop_prob=0.1):
        super().__init__()
        
        # Load SentencePiece tokenizer
        self.tokenizer = spm.SentencePieceProcessor()
        self.tokenizer.Load(tokenizer_path)
        
        # Special token IDs — no manual management needed
        self.PAD_ID   = self.tokenizer.pad_id()   # 0
        self.START_ID = self.tokenizer.bos_id()   # 2
        self.END_ID   = self.tokenizer.eos_id()   # 3
        
        self.vocab_size = self.tokenizer.vocab_size()
        self.max_sequence_length = max_sequence_length
        
        self.embedding = nn.Embedding(self.vocab_size, d_model, padding_idx=self.PAD_ID)
        self.position_encoder = PositionalEncoding(d_model, max_sequence_length)
        self.dropout = nn.Dropout(p=drop_prob)

    def batch_tokenize(self, batch, start_token, end_token):
        
        def tokenize(sentence, start_token, end_token):
            token_ids = self.tokenizer.encode(sentence, out_type=int)  # why out_type = int
            
            if start_token:
                token_ids = [self.START_ID] + token_ids
            if end_token:
                token_ids = token_ids + [self.END_ID]
            
            # Truncate if too long
            token_ids = token_ids[:self.max_sequence_length]
            
            # Pad if too short
            token_ids += [self.PAD_ID] * (self.max_sequence_length - len(token_ids))
            
            return torch.tensor(token_ids, dtype=torch.long)
        
        tokenized = [tokenize(sentence, start_token, end_token) for sentence in batch]
        return torch.stack(tokenized).to(get_device())     # removed to(get_device()) from here

    def forward(self, x, start_token, end_token):
        x = self.batch_tokenize(x, start_token, end_token)  # (batch, seq_len)
        x = self.embedding(x)                                # (batch, seq_len, d_model)
        pos = self.position_encoder()   # removed to(get_device()) from here
        x = self.dropout(x + pos)
        return x


def scaled_dot_product_attention(q, k, v, mask=None):
    # q: 30 x 8 x 200 x 64, k: 30 x 8 x 200 x 64, v: 30 x 8 x 200 x 64, mask 200 x 200
    d_k = q.size()[-1] 
    scaled = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(d_k) # 30 x 8 x 200 x 200
    if mask is not None:
        scaled += mask # 30 x 8 x 200 x 200
    attention = F.softmax(scaled, dim=-1) # 30 x 8 x 200 x 200
    values = torch.matmul(attention, v) # 30 x 8 x 200 x 64
    return values, attention

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, ffn_hidden, drop_prob=0.1):
        super(PositionwiseFeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, ffn_hidden)
        self.linear2 = nn.Linear(ffn_hidden, d_model)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=drop_prob)

    def forward(self, x):
        #  x: 30 x 200 x 512
        x = self.linear1(x) #30 x 200 x 2048
        x = self.relu(x) #30 x 200 x 2048
        x = self.dropout(x) #30 x 200 x 2048
        x = self.linear2(x) #30 x 200 x 512
        return x #30 x 200 x 512
    
class LayerNormalization(nn.Module):
    def __init__(self, parameters_shape, eps=1e-5):
        super().__init__()
        self.parameters_shape=parameters_shape
        self.eps=eps
        self.gamma = nn.Parameter(torch.ones(parameters_shape)) # 512
        self.beta =  nn.Parameter(torch.zeros(parameters_shape)) # 512

    def forward(self, inputs):
        # inputs : 30 x 200 x 512
        dims = [-(i + 1) for i in range(len(self.parameters_shape))] # [-1]
        mean = inputs.mean(dim=dims, keepdim=True) #30 x 200 x 1
        var = ((inputs - mean) ** 2).mean(dim=dims, keepdim=True) # 30 x 200 x 512
        std = (var + self.eps).sqrt() # 30 x 200 x 512
        y = (inputs - mean) / std # 30 x 200 x 512
        out = self.gamma * y  + self.beta  # 30 x 200 x 512
        return out
    
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv_layer = nn.Linear(d_model , 3 * d_model) # 1536 
        self.linear_layer = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        batch_size, sequence_length, d_model = x.size() # 30 x 200 x 512 
        qkv = self.qkv_layer(x) # 30 x 200 x 1536
        qkv = qkv.reshape(batch_size, sequence_length, self.num_heads, 3 * self.head_dim) # 30 x 200 x 8 x 192
        qkv = qkv.permute(0, 2, 1, 3) # 30 x 8 x 200 x 192
        q, k, v = qkv.chunk(3, dim=-1) # q: 30 x 8 x 200 x 64, k: 30 x 8 x 200 x 64, v: 30 x 8 x 200 x 64
        values, attention = scaled_dot_product_attention(q, k, v, mask) # values: 30 x 8 x 200 x 64
        values = values.permute(0,2,1,3).contiguous().reshape(batch_size, sequence_length, self.num_heads * self.head_dim) # 30 x 200 x 512
        out = self.linear_layer(values) # 30 x 200 x 512
        return out # 30 x 200 x 512


NEG_INFTY = -1e9

def create_masks(eng_batch, hi_batch, english_tokenizer, hindi_tokenizer, max_sequence_length):
    num_sentences = len(eng_batch)
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

    # Look ahead mask
    look_ahead_mask = torch.full(
        [num_sentences, max_sequence_length, max_sequence_length], True
    )
    look_ahead_mask = torch.triu(look_ahead_mask, diagonal=1)

    encoder_padding_mask            = torch.full([num_sentences, max_sequence_length, max_sequence_length], False)
    decoder_padding_mask_self_attn  = torch.full([num_sentences, max_sequence_length, max_sequence_length], False)
    decoder_padding_mask_cross_attn = torch.full([num_sentences, max_sequence_length, max_sequence_length], False)

    for idx in range(num_sentences):
        eng_token_len = len(english_tokenizer.encode(eng_batch[idx]))
        hi_token_len  = len(hindi_tokenizer.encode(hi_batch[idx]))

        eng_padding_start = min(eng_token_len + 2, max_sequence_length)
        hi_padding_start  = min(hi_token_len + 2, max_sequence_length)

        eng_padding_cols = torch.arange(eng_padding_start, max_sequence_length)
        hi_padding_cols  = torch.arange(hi_padding_start,  max_sequence_length)

        # Encoder mask
        encoder_padding_mask[idx, :, eng_padding_cols] = True
        encoder_padding_mask[idx, eng_padding_cols, :] = True

        # Decoder self-attention mask
        decoder_padding_mask_self_attn[idx, :, hi_padding_cols] = True
        decoder_padding_mask_self_attn[idx, hi_padding_cols, :] = True

        # Cross-attention mask
        decoder_padding_mask_cross_attn[idx, :, eng_padding_cols] = True
        decoder_padding_mask_cross_attn[idx, hi_padding_cols, :]  = True

    # Convert to additive mask
    encoder_self_attention_mask = torch.where(encoder_padding_mask, NEG_INFTY, 0.0)
    decoder_self_attention_mask = torch.where(
        look_ahead_mask | decoder_padding_mask_self_attn,
        NEG_INFTY, 0.0
    )
    decoder_cross_attention_mask = torch.where(
        decoder_padding_mask_cross_attn,
        NEG_INFTY, 0.0
    )

    #  add head dimension
    encoder_self_attention_mask = encoder_self_attention_mask.unsqueeze(1)
    decoder_self_attention_mask = decoder_self_attention_mask.unsqueeze(1)
    decoder_cross_attention_mask = decoder_cross_attention_mask.unsqueeze(1)

    return (
        encoder_self_attention_mask.to(get_device()),
        decoder_self_attention_mask.to(get_device()),
        decoder_cross_attention_mask.to(get_device())
    )


In [3]:
get_device()

device(type='cpu')

## Encoder ##

In [4]:
'''[encoder.py]'''
        
class EncoderLayer(nn.Module):
    def __init__(self, d_model, ffn_hidden, num_heads, drop_prob):
        super().__init__()
        self.attention = MultiHeadAttention(d_model=d_model,num_heads=num_heads)
        self.norm1 = LayerNormalization(parameters_shape = [d_model])
        self.dropout1 = nn.Dropout(p=drop_prob)
        self.ffn = PositionwiseFeedForward(d_model=d_model,ffn_hidden=ffn_hidden,drop_prob=drop_prob)
        self.norm2 = LayerNormalization(parameters_shape = [d_model])
        self.dropout2 = nn.Dropout(p=drop_prob)

    def forward (self, x, mask=None):
        residual_x = x
        x = self.attention(x,mask=mask)
        x = self.dropout1(x)
        x = self.norm1(x + residual_x)
        residual_x = x
        x = self.ffn(x)
        x = self.dropout2(x)
        x = self.norm2(x + residual_x) 
        return x 
    
class SequentialEncoder(nn.Sequential):
    def forward(self, *inputs):
        x, self_attention_mask  = inputs
        for module in self._modules.values():
            x = module(x, self_attention_mask)
        return x

class Encoder(nn.Module):
    def __init__(self, 
                 d_model, 
                 ffn_hidden, 
                 num_heads, 
                 drop_prob, 
                 num_layers,
                 max_sequence_length,
                 english_tokenizer_path):  
        super().__init__()
        self.sentence_embedding = SentenceEmbedding(
            max_sequence_length, d_model, english_tokenizer_path, drop_prob
        )
        self.layers = SequentialEncoder(*[
            EncoderLayer(d_model, ffn_hidden, num_heads, drop_prob) 
            for _ in range(num_layers)
        ])

    def forward(self, x, self_attention_mask, start_token, end_token):
        x = self.sentence_embedding(x, start_token, end_token)
        x = self.layers(x, self_attention_mask)
        return x

## Decoder ##

In [5]:
'''[decoder.py]'''

class MultiHeadCrossAttention(nn.Module):

    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.kv_layer = nn.Linear(d_model , 2 * d_model) # 1024
        self.q_layer = nn.Linear(d_model , d_model)
        self.linear_layer = nn.Linear(d_model, d_model)
    
    def forward(self, x, y, mask=None):
        batch_size, sequence_length, d_model = x.size() # 30 x 200 x 512
        kv = self.kv_layer(x) # 30 x 200 x 1024
        q = self.q_layer(y) # 30 x 200 x 512
        kv = kv.reshape(batch_size, sequence_length, self.num_heads, 2 * self.head_dim)  # 30 x 200 x 8 x 128
        q = q.reshape(batch_size, sequence_length, self.num_heads, self.head_dim)  # 30 x 200 x 8 x 64
        kv = kv.permute(0, 2, 1, 3) # 30 x 8 x 200 x 128
        q = q.permute(0, 2, 1, 3) # 30 x 8 x 200 x 64
        k, v = kv.chunk(2, dim=-1) # K: 30 x 8 x 200 x 64, v: 30 x 8 x 200 x 64
        values, attention = scaled_dot_product_attention(q, k, v, mask) #  30 x 8 x 200 x 64
        values = values.permute(0,2,1,3).contiguous().reshape(batch_size, sequence_length, d_model) #  30 x 200 x 512
        out = self.linear_layer(values)  #  30 x 200 x 512
        return out  #  30 x 200 x 512
    
class DecoderLayer(nn.Module):

    def __init__(self, d_model, ffn_hidden, num_heads, drop_prob):
        super(DecoderLayer, self).__init__()
        self.self_attention = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
        self.norm1 = LayerNormalization(parameters_shape=[d_model])
        self.dropout1 = nn.Dropout(p=drop_prob)
        self.encoder_decoder_attention = MultiHeadCrossAttention(d_model=d_model, num_heads=num_heads)
        self.norm2 = LayerNormalization(parameters_shape=[d_model])
        self.dropout2 = nn.Dropout(p=drop_prob)
        self.ffn = PositionwiseFeedForward(d_model=d_model, ffn_hidden=ffn_hidden, drop_prob=drop_prob)
        self.norm3 = LayerNormalization(parameters_shape=[d_model])
        self.dropout3 = nn.Dropout(p=drop_prob)

    def forward(self, x, y, self_attention_mask, cross_attention_mask):
        _y = y # 30 x 200 x 512
        y = self.self_attention(y, mask=self_attention_mask) # 30 x 200 x 512
        y = self.dropout1(y) # 30 x 200 x 512
        y = self.norm1(y + _y) # 30 x 200 x 512

        _y = y # 30 x 200 x 512
        y = self.encoder_decoder_attention(x, y, mask=cross_attention_mask) #30 x 200 x 512
        y = self.dropout2(y)
        y = self.norm2(y + _y)  #30 x 200 x 512

        _y = y  #30 x 200 x 512
        y = self.ffn(y) #30 x 200 x 512
        y = self.dropout3(y) #30 x 200 x 512
        y = self.norm3(y + _y) #30 x 200 x 512
        return y #30 x 200 x 512

class SequentialDecoder(nn.Sequential):
    def forward(self, *inputs):
        x, y, self_attention_mask, cross_attention_mask = inputs
        for module in self._modules.values():
            y = module(x, y, self_attention_mask, cross_attention_mask)
        return y

class Decoder(nn.Module):
    def __init__(self, 
                 d_model, 
                 ffn_hidden, 
                 num_heads, 
                 drop_prob, 
                 num_layers,
                 max_sequence_length,
                 hindi_tokenizer_path):  
        super().__init__()
        self.sentence_embedding = SentenceEmbedding(
            max_sequence_length, d_model, hindi_tokenizer_path, drop_prob
        )
        self.layers = SequentialDecoder(*[
            DecoderLayer(d_model, ffn_hidden, num_heads, drop_prob) 
            for _ in range(num_layers)
        ])

    def forward(self, x, y, self_attention_mask, cross_attention_mask, start_token, end_token):
        y = self.sentence_embedding(y, start_token, end_token)
        y = self.layers(x, y, self_attention_mask, cross_attention_mask)
        return y

## Transformer ##

In [6]:
'''[transformer.py]'''

class Transformer(nn.Module):
    def __init__(self, 
                d_model, 
                ffn_hidden, 
                num_heads, 
                drop_prob, 
                num_layers,
                max_sequence_length, 
                english_tokenizer_path,  
                hindi_tokenizer_path,     
                                          
                ):
        super().__init__()
        self.encoder = Encoder(d_model, ffn_hidden, num_heads, drop_prob, num_layers, 
                               max_sequence_length, english_tokenizer_path)
        self.decoder = Decoder(d_model, ffn_hidden, num_heads, drop_prob, num_layers, 
                               max_sequence_length, hindi_tokenizer_path)
        
        # Get hindi vocab size directly from tokenizer
        hindi_tokenizer = spm.SentencePieceProcessor()
        hindi_tokenizer.Load(hindi_tokenizer_path)
        hindi_vocab_size = self.decoder.sentence_embedding.vocab_size
        
        self.linear = nn.Linear(d_model, hindi_vocab_size)
        self.device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

    def forward(self, 
                x, y, 
                encoder_self_attention_mask=None, 
                decoder_self_attention_mask=None, 
                decoder_cross_attention_mask=None,
                enc_start_token=False,
                enc_end_token=False,
                dec_start_token=False,
                dec_end_token=False):
        x = self.encoder(x, encoder_self_attention_mask, enc_start_token, enc_end_token)
        out = self.decoder(x, y, decoder_self_attention_mask, decoder_cross_attention_mask, dec_start_token, dec_end_token)
        out = self.linear(out)
        return out


## Data Processing ##

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path

root = Path("/content/drive/MyDrive/MachineLearning")

In [ ]:
'''spm.SentencePieceTrainer.train(
    input=f"{root}/train.hi",
    model_prefix=f"{root}/hi_tokenizer",
    vocab_size=8000,
    character_coverage=0.9999,
    model_type="bpe",
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
)'''

In [9]:
with open(f"{root}/train.en", 'r', encoding='utf-8') as f:
    english_sentences = [line.strip().lower() for line in f]

with open(f"{root}/train.hi", 'r', encoding='utf-8') as f:
    hindi_sentences = [line.strip() for line in f]

In [ ]:
'''import sentencepiece as spm

en_tok = spm.SentencePieceProcessor()
hi_tok = spm.SentencePieceProcessor()
en_tok.Load(f"{root}/en_tokenizer.model")
hi_tok.Load(f"{root}/hi_tokenizer.model")

en_lengths = [len(en_tok.encode(s)) for s in english_sentences]
hi_lengths = [len(hi_tok.encode(s)) for s in hindi_sentences]

import numpy as np
for name, lengths in [("English", en_lengths), ("Hindi", hi_lengths)]:
    print(f"{name} | max:{max(lengths)} | 95th:{int(np.percentile(lengths, 95))} | 99th:{int(np.percentile(lengths, 99))} | mean:{int(np.mean(lengths))}")'''

In [16]:
english_sentences[:10]

['the bicycle thief',
 'is ricci there?',
 'are you deaf? come on!',
 'get a move on.',
 "and because i'm a bricklayer i should die of hunger?",
 'what do you want from me?',
 'just be patient.',
 "we'll see what we can do.",
 "we'll try to find something.",
 "ricci, you'll hang posters."]

In [17]:
hindi_sentences[:10]

['साइकिल चोर',
 'रिच्ची है क्या?',
 'तुम बहरे हो क्या?',
 'उठो! चलो.',
 'और क्योंकि मैं राजमिस्त्री हूँ मुझे भूख से मर जाना चाहिए?',
 'तुम मुझसे क्या चाहते हो?',
 'बस धैर्य रखो.',
 'हम देखेंगे कि हम क्या कर सकते हैं.',
 'हम कुछ खोजने की कोशिश करेंगे.',
 'रिच्ची, तुम पोस्टर लगाओगे.']

In [10]:
class TextDataset(Dataset):

    def __init__(self, english_sentences, hindi_sentences):
        self.english_sentences = english_sentences
        self.hindi_sentences = hindi_sentences

    def __len__(self):
        return len(self.english_sentences)

    def __getitem__(self, idx):
        return self.english_sentences[idx], self.hindi_sentences[idx]
    
dataset = TextDataset(english_sentences, hindi_sentences)

In [11]:
val_size   = int(0.05 * len(dataset))   # ~3,728
train_size = len(dataset) - val_size    # ~70,841

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

In [12]:
print(f"Train size : {train_size}")
print(f"Val size   : {val_size}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")

Train size : 70841
Val size   : 3728
Train batches: 2214
Val batches  : 117


## Training ##

In [11]:
d_model             = 512
num_layers          = 2
ffn_hidden          = 2048
num_heads           = 8
drop_prob           = 0.1
batch_size          = 16
max_sequence_length = 32

english_tokenizer_path = f"{root}/en_tokenizer.model"
hindi_tokenizer_path = f"{root}/hi_tokenizer.model"

transformer = Transformer(
    d_model=d_model,
    num_layers=num_layers,
    ffn_hidden=ffn_hidden,
    num_heads=num_heads,
    drop_prob=drop_prob,
    max_sequence_length=max_sequence_length,
    english_tokenizer_path=english_tokenizer_path,
    hindi_tokenizer_path=hindi_tokenizer_path
)

In [12]:
model = transformer
def count_params(model):
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if total >= 1_000_000_000:
        return f"{total / 1e9:.2f}B"
    elif total >= 1_000_000:
        return f"{total / 1e6:.2f}M"
    else:
        return f"{total:,}"

print(f"Model size: {count_params(model)}")

Model size: 27.01M


In [30]:
random_ints = torch.randint(0, 101, (1,)).item()
print(random_ints)
l = [torch.randint(0, 101, (1,)).item() for i in range(3)]
l

33


[79, 36, 87]

In [32]:
tokenizer = spm.SentencePieceProcessor()
tokenizer.Load(f"{root}/hi_tokenizer.model")
PAD_ID = tokenizer.pad_id()
criterian = nn.CrossEntropyLoss(ignore_index=PAD_ID, reduction='none')
optim = torch.optim.Adam(transformer.parameters(), lr=1.0)

In [ ]:
device = get_device()
transformer.to(device)

# Hyperparameters
num_epochs    = 10
warmup_steps  = 4000   # lr increases linearly then decays


# ------------------------------------------------------------------
# Warmup scheduler (from "Attention is All You Need" paper)
# lr increases for warmup_steps, then slowly decays
# ------------------------------------------------------------------
def lr_lambda(step):
    step = max(step, 1)  # avoid step=0 division
    # Formula from the original paper
    return (d_model ** -0.5) * min(step ** -0.5, step * warmup_steps ** -1.5)

scheduler = LambdaLR(optim, lr_lambda=lr_lambda)

# ------------------------------------------------------------------
# Training loop
# ------------------------------------------------------------------
for epoch in range(num_epochs):

    transformer.train()
    train_losses = []

    for batch_num, batch in enumerate(train_loader):
        eng_batch, hi_batch = batch

        # --- Create masks for this batch ---
        (enc_mask,
         dec_self_mask,
         dec_cross_mask) = create_masks(
            eng_batch, hi_batch,
            transformer.encoder.sentence_embedding.tokenizer,  # reuse loaded tokenizer
            transformer.decoder.sentence_embedding.tokenizer,
            max_sequence_length
        )

        # Move masks to device
        enc_mask       = enc_mask.to(device)
        dec_self_mask  = dec_self_mask.to(device)
        dec_cross_mask = dec_cross_mask.to(device)

        # --- Forward pass ---
        # Encoder: <start> English sentence <end>
        # Decoder input:  <start> Hindi sentence (teacher forcing, NO end token)
        # Decoder target: Hindi sentence <end>       (NO start token)
        predictions = transformer(
            eng_batch, hi_batch,
            encoder_self_attention_mask = enc_mask,
            decoder_self_attention_mask = dec_self_mask,
            decoder_cross_attention_mask= dec_cross_mask,
            enc_start_token=True,
            enc_end_token=True,
            dec_start_token=True,   # decoder INPUT  starts with <start>
            dec_end_token=False      # decoder INPUT  has no <end>
        )
        # predictions: (batch, seq_len, hindi_vocab_size)

        # --- Build targets (what decoder should have predicted) ---
        # Target = Hindi sentence WITH <end>, WITHOUT <start>
        # We get this by tokenizing with end_token=True, start_token=False
        labels = transformer.decoder.sentence_embedding.batch_tokenize(
            hi_batch, start_token=False, end_token=True
        ).to(device)
        # labels: (batch, seq_len)

        # --- Compute loss ---
        # predictions need to be (batch * seq_len, vocab_size)
        # labels need to be      (batch * seq_len,)
        batch_size_actual = predictions.size(0)
        seq_len           = predictions.size(1)
        vocab_size        = predictions.size(2)

        loss = criterian(
            predictions.view(batch_size_actual * seq_len, vocab_size),
            labels.view(batch_size_actual * seq_len)
        )

        # Mask out padding positions from loss mean
        # (criterian has ignore_index=PAD_ID but reduction='none', so we average manually)
        valid_positions = (labels.view(-1) != PAD_ID).sum()
        loss = loss.sum() / valid_positions

        # --- Backward pass ---
        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(transformer.parameters(), max_norm=1.0)  # prevent exploding gradients
        optim.step()
        scheduler.step()   # update lr every step (not every epoch)

        train_losses.append(loss.item())

        if batch_num % 100 == 0:
            avg = sum(train_losses[-100:]) / len(train_losses[-100:])
            lr  = scheduler.get_last_lr()[0]
            print(f"Epoch {epoch+1} | Batch {batch_num} | Loss: {avg:.4f} | LR: {lr:.6f}")

    # ------------------------------------------------------------------
    # Validation loop (no gradients needed)
    # ------------------------------------------------------------------
    transformer.eval()
    val_losses = []

    with torch.no_grad():
        for batch in val_loader:
            eng_batch, hi_batch = batch

            (enc_mask,
             dec_self_mask,
             dec_cross_mask) = create_masks(
                eng_batch, hi_batch,
                transformer.encoder.sentence_embedding.tokenizer,
                transformer.decoder.sentence_embedding.tokenizer,
                max_sequence_length
            )



            predictions = transformer(
                eng_batch, hi_batch,
                encoder_self_attention_mask = enc_mask,
                decoder_self_attention_mask = dec_self_mask,
                decoder_cross_attention_mask= dec_cross_mask,
                enc_start_token=True,
                enc_end_token=True,
                dec_start_token=True,
                dec_end_token=False
            )

            labels = transformer.decoder.sentence_embedding.batch_tokenize(
                hi_batch, start_token=False, end_token=True
            ).to(device)

            loss = criterian(
                predictions.view(-1, predictions.size(-1)),
                labels.view(-1)
            )
            valid_positions = (labels.view(-1) != PAD_ID).sum()
            loss = loss.sum() / valid_positions
            val_losses.append(loss.item())

    avg_train = sum(train_losses) / len(train_losses)
    avg_val   = sum(val_losses)   / len(val_losses)
    print(f"\nEpoch {epoch+1} DONE | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}\n")
    

    sample_indices = [torch.randint(0, 3728, (1,)).item() for i in range(3)]  #  3 random samples from val set
    transformer.eval()
    with torch.no_grad():
        for idx in sample_indices:
            eng_sentence, hi_sentence = val_dataset[idx]

            # Tokenize encoder input
            enc_input = transformer.encoder.sentence_embedding.batch_tokenize(
                [eng_sentence], start_token=True, end_token=True
            ).to(device)  # (1, seq_len)

            # Encode
            enc_mask = torch.zeros(1, 1, max_sequence_length, max_sequence_length).to(device)
            encoder_output = transformer.encoder.layers(
                transformer.encoder.sentence_embedding(
                    [eng_sentence], start_token=True, end_token=True
                ),
                enc_mask
            )  # (1, seq_len, d_model)

            # Greedy decode — generate Hindi token by token
            hi_tokenizer = transformer.decoder.sentence_embedding.tokenizer
            START_ID = hi_tokenizer.bos_id()
            END_ID   = hi_tokenizer.eos_id()

            generated = [START_ID]  # start with <start> token

            for _ in range(max_sequence_length):
                # Decode so far
                dec_input_ids = torch.tensor([generated], dtype=torch.long).to(device)  # (1, current_len)

                # Pad to max_sequence_length
                pad_len = max_sequence_length - len(generated)
                dec_input_padded = torch.nn.functional.pad(dec_input_ids, (0, pad_len))  # (1, seq_len)

                # Get embedding directly from embedding layer (bypass batch_tokenize)
                dec_emb = transformer.decoder.sentence_embedding.embedding(dec_input_padded)
                pos     = transformer.decoder.sentence_embedding.position_encoder()
                dec_emb = dec_emb + pos  # (1, seq_len, d_model)

                # No mask for inference (or simple causal mask)
                dec_mask = torch.zeros(1, 1, max_sequence_length, max_sequence_length).to(device)

                # Run decoder layers
                dec_output = transformer.decoder.layers(encoder_output, dec_emb, dec_mask, dec_mask)

                # Linear projection to vocab
                logits = transformer.linear(dec_output)  # (1, seq_len, vocab_size)

                # Pick token at current position
                next_token = logits[0, len(generated) - 1, :].argmax(dim=-1).item()

                if next_token == END_ID:
                    break

                generated.append(next_token)

            # Decode token ids to string
            predicted_hindi = hi_tokenizer.decode(generated[1:])  # skip <start>

            print(f"  EN : {eng_sentence}")
            print(f"  HI (true)     : {hi_sentence}")
            print(f"  HI (predicted): {predicted_hindi}")
            print()

    # ------------------------------------------------------------------
    # Save checkpoint after every epoch
    # ------------------------------------------------------------------
    torch.save({
        'epoch'               : epoch + 1,
        'model_state_dict'    : transformer.state_dict(),
        'optimizer_state_dict': optim.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss'          : avg_train,
        'val_loss'            : avg_val,
    }, f"{root}/checkpoint_epoch_{epoch+1}.pt")
    print(f"Checkpoint saved for epoch {epoch+1}")